# Paper 1 — Multi-seed experiments (Chicago benchmark)
Runs 8 spatio-temporal models (HA, LSTM, STGCN, GraphWaveNet, DCRNN, MTGNN, InformerOnly, HASI-Net) across 5 seeds plus the component ablation, on the Chicago 4-crime panel. PSO hyperparameter search runs once and is frozen across seeds. Every result is written from a real run; nothing is fabricated.

GPU runtime: Runtime → Change runtime type → T4 GPU.

## 0. Bootstrap (clone repo, mount Drive for results)

In [ ]:
# Bootstrap: deps + clone repo from GitHub + Drive for persistent results.
import subprocess, sys, os, pathlib
def pip(p): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])
for p in ["torch", "numpy", "pandas", "requests", "geopandas", "shapely",
          "matplotlib", "pyarrow", "scipy"]:
    try: __import__(p)
    except ImportError: pip(p)

# Clone (or pull if stale) the public repo to the ephemeral Colab filesystem.
REPO = pathlib.Path("/content/spatio-temporal-crime")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q",
        "https://github.com/vivekjindal24/spatio-temporal-crime.git", str(REPO)],
        check=True)
sys.path.insert(0, str(REPO / "src"))
os.chdir(str(REPO))

# Mount Drive and route results there so they survive Colab disconnects.
# (The package reads HASI_RESULTS_DIR at import time.)
from google.colab import drive
drive.mount("/content/drive")
os.environ["HASI_RESULTS_DIR"] = "/content/drive/MyDrive/spatio-temporal-crime_results"
print("Repo:", REPO, "| results ->", os.environ["HASI_RESULTS_DIR"])


In [ ]:
# Config (Paper 1, Chicago monthly).
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
import pathlib; pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
cfg = Config(
    target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
    chicago_year_start=2015, chicago_year_end=2024,
    device="cuda", lookback=12, horizon=3,
    epochs=80, batch_size=64, lr=1e-3,
    hidden_dim=64, n_graph_layers=2, n_attn_heads=4,
    loss_type="log1p_mse", pso_enabled=True)
print("Results dir:", RESULTS_DIR)
print("Config:"); print(cfg.to_dict())

## 1. Build the Chicago panel (downloads ~775k incidents on first run, cached thereafter)

In [ ]:
from hasi_net.data import build_chicago_panel
panel = build_chicago_panel(cfg)
print("Counts [T, N, C]:", panel.counts.shape)
print("Categories:", panel.categories)
import numpy as np
for j, c in enumerate(panel.categories):
    print(f"  {c:24s} mean {panel.counts[...,j].mean():.2f} "
          f"zero {(panel.counts[...,j]==0).mean()*100:.1f}%")

## 2. Multi-seed run: 8 models x 5 seeds (PSO once, frozen)

In [ ]:
from hasi_net.multiseed import run_multiseed
SEEDS = [42, 1, 2, 3, 4]
meanstd, perseed = run_multiseed("chicago", cfg, seeds=SEEDS,
                                 tag="p1_chicago", pso=True, verbose=True)
meanstd.round(4)

## 3. Component ablation x 5 seeds

In [ ]:
from hasi_net.multiseed import run_ablation_multiseed
abl = run_ablation_multiseed("chicago", cfg, seeds=SEEDS, tag="p1_chicago",
                             epochs=40, verbose=True)
abl.round(4)

## 4. Persist results to Drive
All CSVs/JSONs/PNGs are written to the Drive results dir by the package (see `RESULTS_DIR`), so they survive Colab disconnects.

In [ ]:
# Verify P1 artifacts in the Drive results dir.
from hasi_net.config import RESULTS_DIR
import pathlib
arts = sorted(p.name for p in pathlib.Path(RESULTS_DIR).glob("*p1_chicago*"))
print("P1 artifacts in", RESULTS_DIR)
for a in arts: print(" ", a)
